In [159]:
import sys
PATH_TO_PREDICTOR_HOME = ".."
sys.path.append(PATH_TO_PREDICTOR_HOME)

from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS


# 0. Read the data

In [160]:
from config import PER_ORGANISM_PER_CLASS_ALLELES, RESULT_COLUMNS
DATA_PATH = "/home/denklewer/Data/schreiber_full/"
# read total data

total_df =  pd.read_csv("{}schreiber_total_data_full.csv".format(DATA_PATH),sep=";")
ALLELES = list(set(total_df['allele']))
per_allele_mt_df =dict()
per_allele_wt_df = dict()

for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    df_mt = pd.read_csv("{}schreiber_data_full_{}_mt.csv".format(DATA_PATH,target_allele_name),sep=";")
    df_wt = pd.read_csv("{}schreiber_data_full_{}_wt.csv".format(DATA_PATH,target_allele_name),sep=";") 
    
    per_allele_mt_df[allele_name] = df_mt
    per_allele_wt_df[allele_name] = df_wt
    
    assert 'peptide' in df_mt.columns
    assert 'length' in df_mt.columns
    assert RESULT_COLUMNS['id'] in df_mt.columns
    assert 'peptide' in df_wt.columns
    assert 'length' in df_wt.columns
    assert RESULT_COLUMNS['id'] in df_wt.columns

    
def defineOrganism(allele_name):
    allele_name = allele_name.replace('H-2', 'H2')
    for organism_to_use in PER_ORGANISM_PER_CLASS_ALLELES.keys():
        per_class_alleles = PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use]
        for class_to_use, alleles in per_class_alleles.items():
            if allele_name in alleles:
                return organism_to_use
    print("organism not  found", allele_name)
    return None

    
def defineClass(allele_name):
    organism =  defineOrganism(allele_name)
    allele_name = allele_name.replace('H-2', 'H2')
    
    per_class_alleles = PER_ORGANISM_PER_CLASS_ALLELES[organism]
    for class_to_use, alleles in per_class_alleles.items():
        if allele_name in alleles:
            return class_to_use
    print("not found")
    return None

CLASSES_TO_USE = list(set([defineClass(allele_name) for allele_name in ALLELES]))
ORGANISMS_TO_USE = list(set([defineOrganism(allele_name).lower() for allele_name in ALLELES]))

    
ALLELES, CLASSES_TO_USE, ORGANISMS_TO_USE

(['H-2-Dd', 'H-2-IEd', 'H-2-Ld', 'H-2-Kd', 'H-2-IAd'], ['I', 'II'], ['mouse'])

In [161]:
per_method_per_allele_wt_predictions = dict()
per_method_per_allele_mt_predictions = dict()

In [162]:
METHOD = "ghmm"

## 0.1. make result dicts

In [163]:
per_method_per_allele_wt_predictions = dict()
per_method_per_allele_mt_predictions = dict()

## 0.2 Read decoy data

In [164]:
per_allele_decoy = dict()

for allele_name in ALLELES:
    organism =  defineOrganism(allele_name).lower()
    class_to_use = defineClass(allele_name)
    DECOY_DATA_PATH = "/home/denklewer/Data/decoy/sampled_decoy_data_{}_class_{}.csv".format(organism, class_to_use)
    decoy_df =  pd.read_csv(DECOY_DATA_PATH)
    decoy_df['length'] = decoy_df['peptide'].str.len()
    decoy_df['allele'] = allele_name
    
    per_allele_decoy[allele_name] = decoy_df.copy()
[len(decoy_df) for decoy_df in per_allele_decoy.values()]

[200000, 200000, 200000, 200000, 200000]

# 1. Predict with ghmm

In [142]:
import numpy as np
import pandas as pd
import ghmm
import os
import logging
import sys
from scipy.version import full_version as spversion

from __future__ import print_function
LOG_DIR = '../logs/'

if not os.path.exists(LOG_DIR):
    os.makedirs(LOG_DIR)
    
logging.basicConfig(
    filename='{}{}_train.log'.format(LOG_DIR,METHOD), 
    filemode='a',
    encoding='utf-8', 
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)-1s: %(message)s',
            datefmt='%d-%m-%y %H:%M:%S') 
logger = logging.getLogger("GHMM")
logger.setLevel(logging.CRITICAL)

In [143]:
logger.info('Python    :{}'.format(sys.version))
logger.info('Numpy     :{}'.format(np.__version__))
logger.info("Scipy     :{}".format(spversion))
logger.info('Pandas    :{}'.format(pd.__version__))

logger.info("GHMM:      0.9 (manually tracked)") # no such thing as ghmm.__version__, and not under pip
logger.info('Workdir   :{}'.format(os.getcwd()))

In [166]:
# different allele naming.
TARGET_ALLELES = dict()

## HARDCODE
for allele_name in ALLELES:
    if allele_name == 'H2_d class II':
        target_allele_name = 'H2_IAd'
    elif allele_name == 'H2_b class II':
        target_allele_name = 'H2_IAb'
    elif allele_name == 'HLA_DRB1_01_01':
        target_allele_name = 'HLA-DRB101:01'
    elif allele_name == 'HLA_DRB1_04_01':
        target_allele_name = 'HLA-DRB104:01'
    elif allele_name == 'HLA_DRB1_04_05':
        target_allele_name = 'HLA-DRB104:05'    
    else:
        target_allele_name = normalize_allele_name(allele_name).replace("*", "").replace("-", "").lower()
    TARGET_ALLELES[allele_name] = target_allele_name


TARGET_ALLELES

{'H-2-Dd': 'h2dd',
 'H-2-IAd': 'h2iad',
 'H-2-IEd': 'h2ied',
 'H-2-Kd': 'h2kd',
 'H-2-Ld': 'h2ld'}

In [167]:
MODEL_PATH = '../trained_models/old_ghmm/'
#supported_alleles = ['h2iab', 'h2iad', 'iag7', 'h2iak', 'h2iaq','h2ias', 'h2iau', 'h2ied', 'h2iek' ] 
supported_alleles = ['h2ld', 'h2kd', 'h2kb', 'h2iek', 'h2ied', 
                     'h2iau', 'h2ias', 'h2iaq', 'h2iak', 'h2iag7',
                     'h2iad', 'h2iab', 'h2dd', 'h2db'] 
        
        
        
AminoAcidsZ = ghmm.Alphabet(ghmm.AminoAcids.listOfCharacters + ['Z'])

def prepareHmmSequenceSets(peptides, stopEmission=True):
    if stopEmission:
        peptideList = [str(p) + 'Z' for p in peptides]
    else:
        peptideList = [str(p) for p in peptides]
    
    # need to split in blocks in case there are more than 1.5M peptides (ghmm limitation)
    ghmmMaxSequencesPerSet = 1500000
    lenPeptides = len(peptides)
    numFullBlocks = lenPeptides // ghmmMaxSequencesPerSet
    lenLastBlock = lenPeptides % ghmmMaxSequencesPerSet
    sequenceSets = []
    
    # full blocks (if any)
    for i in range(0, numFullBlocks):
        rangeStart = i * ghmmMaxSequencesPerSet
        rangeEnd = rangeStart + ghmmMaxSequencesPerSet
        sequenceBlock = ghmm.SequenceSet(
            AminoAcidsZ,
            [list(p) for p in peptideList[rangeStart:rangeEnd]]
        )
        sequenceSets.append(sequenceBlock)
    rangeStart = numFullBlocks * ghmmMaxSequencesPerSet
    rangeEnd = lenPeptides
    if (lenLastBlock > 0):
        sequenceBlock = ghmm.SequenceSet(
                AminoAcidsZ,
                [list(p) for p in peptideList[rangeStart:rangeEnd]]
            )
        sequenceSets.append(sequenceBlock)
    return sequenceSets

def predictHmmPeptidesRaw(hmmModelFile, sequenceSets):
    """Predict -10logp values using a given HMM model for binders and non-binders.
       Results will contain -inf values
    """
    hmm = ghmm.HMMOpen(hmmModelFile)

    return np.concatenate(
        [-10 * np.array(hmm.loglikelihoods(s)) for s in sequenceSets]
    )


def predict_with_ghmm(per_allele_df):
    per_allele_predictions = dict()
    for allele_name in ALLELES:
        print(allele_name)
        target_allele = TARGET_ALLELES[allele_name]
        df_val = per_allele_df[allele_name].copy()
        if target_allele in supported_alleles:
            sequenceSet = prepareHmmSequenceSets(df_val.peptide)
            print("Predict with model")
            print("Binders")
            df_val['HMMB'] = predictHmmPeptidesRaw('{}{}-splitwithdecoy-binders.xml'.format(MODEL_PATH, target_allele),sequenceSet)

            columnsHMM = ['HMMB']
            df_val.loc[:, columnsHMM] = df_val.loc[:, columnsHMM].div(df_val.length, axis=0)
            df_val = df_val.replace(np.inf, 10000.0)
        else:
            print("Allele {} is not supported, create dummy predictions".format(target_allele))
            df_val['HMMB'] = np.nan
        per_allele_predictions[allele_name] = df_val
        
    
    return per_allele_predictions

def update_scores_with_percentiles(per_allele_scores, per_allele_ref_scores, score_name):
    for allele_name in per_allele_scores.keys():
        df = per_allele_scores[allele_name].copy()
        ref_df =  per_allele_ref_scores[allele_name].copy()
        scores = df[score_name]
        ref_scores = ref_df[score_name]
        ref_scores = ref_scores.sort_values()
        pct_increment = 1./len(ref_scores)
        percentile_column_name = 'percentile_rank'
        percentile_ranks = (0.5 + 0.5*ref_scores.searchsorted(scores, side='left') + 0.5*ref_scores.searchsorted(scores, side='right'))*pct_increment
        percentile_ranks = percentile_ranks*100
        per_allele_scores[allele_name][percentile_column_name] = percentile_ranks
    return per_allele_scores
        






    

# Calculate for decoy

In [168]:
%%time
print("Decoy")
per_allele_decoy_result = predict_with_ghmm(per_allele_decoy)


Decoy
H-2-Dd
Allele h2dd is not supported, create dummy predictions
H-2-IEd
Total peptides: 200000
Full blocks: 0
Peptides in the last block: 200000
Total blocks: 1
Predict with model
Binders
H-2-Ld
Allele h2ld is not supported, create dummy predictions
H-2-Kd
Allele h2kd is not supported, create dummy predictions
H-2-IAd
Total peptides: 200000
Full blocks: 0
Peptides in the last block: 200000
Total blocks: 1
Predict with model
Binders
CPU times: user 24.7 s, sys: 210 ms, total: 24.9 s
Wall time: 24.8 s


In [169]:
%%time
print("--mt")
per_allele_mt_predictions = predict_with_ghmm(per_allele_mt_df)
per_allele_mt_predictions = update_scores_with_percentiles(per_allele_mt_predictions, per_allele_decoy_result, 'HMMB' )

per_method_per_allele_mt_predictions[METHOD] = per_allele_mt_predictions  
print("--wt")
per_allele_wt_predictions = predict_with_ghmm(per_allele_wt_df)
per_allele_wt_predictions = update_scores_with_percentiles(per_allele_wt_predictions, per_allele_decoy_result, 'HMMB' )

per_method_per_allele_wt_predictions[METHOD] = per_allele_wt_predictions  


--mt
H-2-Dd
Allele h2dd is not supported, create dummy predictions
H-2-IEd
Total peptides: 52091
Full blocks: 0
Peptides in the last block: 52091
Total blocks: 1
Predict with model
Binders
H-2-Ld
Allele h2ld is not supported, create dummy predictions
H-2-Kd
Allele h2kd is not supported, create dummy predictions
H-2-IAd
Total peptides: 52091
Full blocks: 0
Peptides in the last block: 52091
Total blocks: 1
Predict with model
Binders
--wt
H-2-Dd
Allele h2dd is not supported, create dummy predictions
H-2-IEd
Total peptides: 52091
Full blocks: 0
Peptides in the last block: 52091
Total blocks: 1
Predict with model
Binders
H-2-Ld
Allele h2ld is not supported, create dummy predictions
H-2-Kd
Allele h2kd is not supported, create dummy predictions
H-2-IAd
Total peptides: 52091
Full blocks: 0
Peptides in the last block: 52091
Total blocks: 1
Predict with model
Binders
CPU times: user 12.8 s, sys: 265 ms, total: 13.1 s
Wall time: 12.7 s


# 2. Join predictions

In [171]:
per_method_columns = {
    'netMHC-3.4': ['source_sequence_name' ,'score', 'affinity', 'percentile_rank'],
    'netMHC-4.0':['source_sequence_name', 'score', 'affinity', 'percentile_rank' ],
    'netMHCpan-4.1':['source_sequence_name', 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'netMHCIIpan-4.0':['source_sequence_name', 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'mhcnuggets':['source_sequence_name', 'score', 'affinity', 'percentile_rank'],
    'ghmm':['source_sequence_name', 'HMMB', 'percentile_rank']

}


def rename_columns(df, method):
    leave_columns = ['source_sequence_name', 'peptide']
    old_columns = [col for col in per_method_columns[method] if col not in leave_columns]
    new_columns = [method + "_" + col for col in old_columns]
    df = df.rename(columns={old:new for old,new in zip(old_columns,new_columns)})
    return df


def getPredictionResult(total_results, allele_name, method):
    df = total_results[method][allele_name][per_method_columns[method]].copy()
    df = rename_columns(df, method)
    return df
# def getPredictionResult(total_results, region, allele_name, method):
#     df = total_results[region][method][allele_name][per_method_columns[method]].copy()
#     df = rename_columns(df, method)
#     return df


In [172]:
def joinPredictionsToSource(per_region_per_method_per_allele_mt_predictions, per_region_per_method_per_allele_wt_predictions):
    per_allele_result = dict()
    per_allele_result = dict()
    methods = list(per_method_per_allele_mt_predictions.keys())
    for allele_name in ALLELES:
        print("allele", allele_name)
        df_source_wt = per_allele_wt_df[allele_name].copy()
        df_source_mt = per_allele_mt_df[allele_name].copy()

        for method in methods:
            print("method", method)
            df_method_wt = getPredictionResult(per_method_per_allele_wt_predictions, allele_name, method)
            df_method_mt = getPredictionResult(per_method_per_allele_mt_predictions, allele_name, method)
            df_source_wt = df_source_wt.merge(df_method_wt, on=['source_sequence_name'], how='left')
            df_source_mt = df_source_mt.merge(df_method_mt, on=['source_sequence_name'], how='left')
        leave_columns = [ 'source_sequence_name', 'allele', 'length']
        df_source_wt = df_source_wt.rename(columns={col:  col + "_wt" for col in df_source_wt.columns if col not in leave_columns})
        df_source_mt = df_source_mt.rename(columns={col:  col + "_mt" for col in df_source_mt.columns if col not in leave_columns})
        # rename id back 


        df_result = df_source_mt.merge(df_source_wt, on=['source_sequence_name', 'allele', 'length'], how='left')
        per_allele_result[allele_name] = df_result
    return per_allele_result
                

              

In [173]:
per_allele_result = joinPredictionsToSource(per_method_per_allele_mt_predictions, per_method_per_allele_wt_predictions)

allele H-2-Dd
method ghmm
allele H-2-IEd
method ghmm
allele H-2-Ld
method ghmm
allele H-2-Kd
method ghmm
allele H-2-IAd
method ghmm


# 5. For now just write to files

In [174]:
TARGET_PATH_TO_PREDICTIONS = DATA_PATH + 'predictions/{}/'.format(METHOD)

import os
if not os.path.exists(TARGET_PATH_TO_PREDICTIONS):
    os.makedirs(TARGET_PATH_TO_PREDICTIONS)

In [175]:
# for decoy_key in DECOY_PREFIXES:

for allele_name in ALLELES:
    total_df = per_allele_result[allele_name]
    allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    total_df.to_csv("{}{}_predictions_allele_{}.csv".format(TARGET_PATH_TO_PREDICTIONS, METHOD,allele_name),sep=";", index=False)

In [176]:
per_allele_result['H-2-IEd']['ghmm_HMMB_mt'].max()

10000.0

In [177]:
total_df['ghmm_HMMB_mt'].max()

41.707389020211615

In [178]:
total_df

,source_sequence_name,allele,peptide_mt,length,ghmm_HMMB_mt,ghmm_percentile_rank_mt,peptide_wt,ghmm_HMMB_wt,ghmm_percentile_rank_wt
0,seq11,H-2-IAd,EGAERGLSAIRER,13,25.441927,0.07275,EGAERGVSAIRER,25.401205,0.06775
1,seq14,H-2-IAd,QPAAAPAAALSAAD,14,24.795878,0.03675,QPPAAPAAALSAAD,25.174292,0.05375
2,seq16,H-2-IAd,EGAERGLSAIRERL,14,25.397658,0.06775,EGAERGVSAIRERL,25.368416,0.06575
3,seq17,H-2-IAd,QPAAAPAAALSAA,13,24.944689,0.04275,QPPAAPAAALSAA,25.691868,0.10125
4,seq18,H-2-IAd,REGAERGLSAIRER,14,26.330534,0.21475,REGAERGVSAIRER,26.299534,0.20725
5,seq21,H-2-IAd,TQQPAAAPAAALSAA,15,24.311675,0.02125,TQQPPAAPAAALSAA,26.102447,0.16075
6,seq22,H-2-IAd,QQPAAAPAAALSAA,14,24.734588,0.03225,QQPPAAPAAALSAA,26.369131,0.22525
7,seq23,H-2-IAd,TQQPAAAPAAALSAAD,16,24.481536,0.02625,TQQPPAAPAAALSAAD,25.692167,0.10125
8,seq24,H-2-IAd,QQPAAAPAAALSAAD,15,24.845232,0.03975,QQPPAAPAAALSAAD,25.855557,0.12025
9,seq25,H-2-IAd,EGAERGLSAIRE,12,26.135309,0.16925,EGAERGVSAIRE,26.099344,0.15950
